# Directed and Weighted Centrality

A variety of measures can be applied to assess **centrality** or node importance in networks that have directed and/or weighted connections between nodes. 

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.options.display.float_format = '{:.3f}'.format

In this notebook, we will use data from the widely-studied *Enron* email corpus, which is described [here](https://en.wikipedia.org/wiki/Enron_Corpus). This dataset provides a real-world example of directed, weighted network relationships.

First, we load the data and construct a network representing the internal Enron email communication system. Each line in our input *edge list* file contains a sender email address, a recipient email address, and the number of emails transmitted from the sender to the recipient during a given time period.

It is appropriate to represent this data as a directed network, with edge weights indicating the number of emails sent from each sender to each recipient.

In [ ]:
# load the lines from the edge list
fin = open("enron.edgelist", "r")
lines = fin.readlines()
fin.close()

In [ ]:
# create an edge based on edge line in the edge list file
g = nx.DiGraph()
for line in lines:
    parts = line.strip().split("\t")
    num_emails = int(parts[2])
    # the weight on the edge is the number of emails from sender to recipient
    g.add_edge(parts[0], parts[1], weight=num_emails)

In [ ]:
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} weighted edges")

## Unweighted In-Degrees and Out-Degrees

First, we examine the simple count of incoming edges - the **unweighted in-degree**. For email communication, this metric indicates the number of unique individuals from whom each person has received at least one email during the observation period, regardless of the total volume of emails received.

In [ ]:
# get a dictionary of in-degree scores for all nodes
in_degrees = dict(g.in_degree())
in_degrees

Examine the statistical properties and distribution of unweighted in-degree values to understand the communication patterns:

In [ ]:
indeg = pd.Series(in_degrees)
print(f"In-degree range: [{indeg.min()}, {indeg.max()}]")
print(f"Mean in-degree: {indeg.mean():.2f}")
print(f"Median in-degree: {indeg.median():.0f}")

In [ ]:
ax = indeg.plot.hist(figsize=(9, 5), fontsize=12, legend=None, color="darkred", bins=20, zorder=3)
ax.yaxis.grid()
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Unweighted In-Degree", fontsize=12)
plt.show()

Identify the top 10 nodes ranked by in-degree. These represent the individuals who received emails from the largest number of unique senders within the company:

In [ ]:
indeg.sort_values(ascending=False).head(10)

Next, we examine the simple count of outgoing edges - the **unweighted out-degree**. This corresponds to the number of unique individuals to whom each person has sent at least one email during the observation period, providing insight into the breadth of each person's communication network.

In [ ]:
# get a dictionary of out-degree scores for all nodes
out_degrees = dict(g.out_degree())

We observe that the minimum out-degree is 0, indicating that these nodes represent individuals who sent no emails during the time period covered by this dataset. 

In [ ]:
outdeg = pd.Series(out_degrees)
print(f"Out-degree range: [{outdeg.min()}, {outdeg.max()}]")
print(f"Mean out-degree: {outdeg.mean():.2f}")
print(f"Median out-degree: {outdeg.median():.0f}")

In [ ]:
ax = outdeg.plot.hist(figsize=(9, 5), fontsize=12, legend=None, color="darkgreen", bins=20, zorder=3)
ax.yaxis.grid()
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Unweighted Out-Degree", fontsize=12)
plt.show()

Identify the top 10 nodes ranked by out-degree. These represent the individuals who sent emails to the largest number of unique email addresses:

In [ ]:
outdeg.sort_values(ascending=False).head(10)

## Weighted In-Degrees and Out-Degrees

So far, we have not considered the actual volume of emails sent between each pair of employees, focusing only on the existence of connections. We can now examine weighted degrees, which consider email counts to provide a more comprehensive view of communication intensity.

Calculate the weighted in-degree, which represents the total number of emails received by each employee across all their correspondents:

In [ ]:
# get a dictionary of in-degree scores for all nodes, using values from the "weight" attribute
win_degrees = dict(g.in_degree(weight="weight"))

We can see from the statistical summary that the range of values becomes substantially larger when we incorporate edge weights, reflecting the significant variation in email communication intensity between employees:

In [ ]:
windeg = pd.Series(win_degrees)
print(f"Weighted in-degree range: [{windeg.min()}, {windeg.max()}]")
print(f"Mean weighted in-degree: {windeg.mean():.2f}")
print(f"Median weighted in-degree: {windeg.median():.0f}")

In [ ]:
ax = windeg.plot.hist(figsize=(9, 5), fontsize=12, legend=None, color="darkred", bins=20, zorder=3)
ax.yaxis.grid()
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Weighted In-Degree", fontsize=12)
plt.show()

We could check which employees received the highest total volume of emails within the company during this time period:

In [ ]:
windeg.sort_values(ascending=False).head(10)

An analogous measure, **weighted out-degree**, is calculated based on the number of edges emanating from a node, weighted by the intensity of each connection. This corresponds to the total number of emails sent by each individual across all their recipients.

In [ ]:
# get a dictionary of weighted out-degree scores for all nodes
wout_degrees = dict(g.out_degree(weight="weight"))

In [ ]:
woutdeg = pd.Series(wout_degrees)
print(f"Weighted out-degree range: [{woutdeg.min()}, {woutdeg.max()}]")
print(f"Mean weighted out-degree: {woutdeg.mean():.2f}")
print(f"Median weighted out-degree: {woutdeg.median():.0f}")

In [ ]:
ax = woutdeg.plot.hist(figsize=(9, 5), fontsize=12, legend=None, color="darkgreen", bins=20, zorder=3)
ax.yaxis.grid()
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Weighted Out-Degree", fontsize=12)
plt.show()

From the distribution plot and the node rankings, we see that one user accounts for 38% of all emails sent during this period, representing an exceptionally high level of communication activity:

In [ ]:
woutdeg.sort_values(ascending=False).head(10)

In [ ]:
# calculate percentage from this user
100.0 * (woutdeg["pete.davis@enron.com"]/woutdeg.sum())

If we remove this exceptionally active individual from the analysis, we can get a clearer visualisation of the distribution of out-degree scores for the remainder of the employees, revealing more typical communication patterns:

In [ ]:
woutdeg2 = woutdeg.drop("pete.davis@enron.com")
ax = woutdeg2.plot.hist(figsize=(9, 5), fontsize=12, legend=None, color="darkgreen", bins=20, zorder=3)
ax.yaxis.grid()
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Weighted Out-Degree", fontsize=12)
plt.show()

## Other Weighted Centrality Measures

NetworkX provides implementations of various other centrality measures that can incorporate edge weights, providing more complex analyses of node importance in weighted networks. 

For example, we can compute **weighted eigenvector centrality**, which incorporates the strength of connections by specifying the attribute to use for edge weights. This approach considers not only the number of connections but also their relative importance:

In [ ]:
w_eigs = dict(nx.eigenvector_centrality(g, weight="weight"))
# convert dictionary to a series
weig = pd.Series(w_eigs)
# get top 20
weig.sort_values(ascending=False).head(20)

Similarly, we can calculate **weighted betweenness centrality**, where shortest paths are computed taking into account edge weights. This provides a more nuanced understanding of which nodes serve as important intermediaries in the weighted network structure:

In [ ]:
w_bets = dict(nx.betweenness_centrality(g, weight="weight"))
# convert dictionary to a series
wbet = pd.Series(w_bets)
# get top 20
wbet.sort_values(ascending=False).head(20)